# Full Project Runner

This notebook reruns the implemented Hybrid RAG QA project from one place. It calls the existing project scripts and checks; it does not recreate source files or replace the individual team notebooks.

Run the cells from top to bottom to verify DVC data access, code quality, BM25 retrieval, retrieval evaluation, semantic search, RAG generation, citation-quality files, and the Gradio callback.

## 1. Locate Project and Define Helpers

In [1]:
from __future__ import annotations

import importlib.util
import json
from pathlib import Path
import subprocess
import sys
import time

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "README.md").exists():
    raise RuntimeError("Run this notebook from inside the project repository.")

PYTHON = sys.executable
DATASET_PATH = PROJECT_ROOT / "data" / "stacklite_questions.csv"
DVC_PATH = PROJECT_ROOT / "data" / "stacklite_questions.csv.dvc"
RUN_RESULTS: list[dict[str, object]] = []

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {PYTHON}")
print(f"Dataset exists: {DATASET_PATH.exists()} ({DATASET_PATH})")

Project root: C:\Users\RTX\Downloads\Advanced DL
Python executable: C:\Users\RTX\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\python.exe
Dataset exists: True (C:\Users\RTX\Downloads\Advanced DL\data\stacklite_questions.csv)


In [2]:
def run_command(name: str, command: list[str], timeout: int = 900) -> subprocess.CompletedProcess[str]:
    """Run a project command, print captured output, and stop on failure."""

    command = [str(part) for part in command]
    start = time.perf_counter()
    print("\n" + "=" * 88)
    print(name)
    print("$ " + " ".join(command))
    print("=" * 88)

    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    elapsed = time.perf_counter() - start
    output = completed.stdout.strip()
    if output:
        print(output)

    status = "PASS" if completed.returncode == 0 else "FAIL"
    print(f"\n{name}: {status} in {elapsed:.1f}s")
    RUN_RESULTS.append(
        {
            "step": name,
            "status": status,
            "return_code": completed.returncode,
            "seconds": round(elapsed, 1),
        }
    )

    if completed.returncode != 0:
        raise RuntimeError(f"{name} failed with exit code {completed.returncode}")
    return completed


def missing_modules(module_names: list[str]) -> list[str]:
    return [name for name in module_names if importlib.util.find_spec(name) is None]

## 2. Dependency and Dataset Check

In [3]:
project_modules = ["pandas", "sentence_transformers", "transformers", "gradio"]
dev_modules = ["pytest", "pytest_cov", "ruff"]

if missing := missing_modules(project_modules):
    print(f"Missing project modules: {missing}")
    run_command("Install project dependencies", [PYTHON, "-m", "pip", "install", "-r", "requirements.txt"], timeout=900)
else:
    print("Project dependencies are available.")

if missing := missing_modules(dev_modules):
    print(f"Missing development modules: {missing}")
    run_command("Install development dependencies", [PYTHON, "-m", "pip", "install", "-r", "requirements-dev.txt"], timeout=600)
else:
    print("Development dependencies are available.")

if importlib.util.find_spec("dvc") is None:
    run_command("Install DVC", [PYTHON, "-m", "pip", "install", "dvc"], timeout=900)
else:
    print("DVC is available.")

Project dependencies are available.
Development dependencies are available.
DVC is available.


In [4]:
if DATASET_PATH.exists():
    print("Dataset CSV is already available locally.")
elif DVC_PATH.exists():
    run_command("Pull DVC dataset", [PYTHON, "-m", "dvc", "pull", str(DVC_PATH.relative_to(PROJECT_ROOT))], timeout=900)
else:
    raise FileNotFoundError("Neither data/stacklite_questions.csv nor its .dvc pointer was found.")

if not DATASET_PATH.exists():
    raise FileNotFoundError("DVC pull finished, but data/stacklite_questions.csv is still missing.")

print(f"Dataset ready: {DATASET_PATH.relative_to(PROJECT_ROOT)}")
print(f"Dataset size: {DATASET_PATH.stat().st_size:,} bytes")
run_command("DVC status", [PYTHON, "-m", "dvc", "status"], timeout=300)

Dataset CSV is already available locally.
Dataset ready: data\stacklite_questions.csv
Dataset size: 1,835,446 bytes

DVC status
$ C:\Users\RTX\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\python.exe -m dvc status
Data and pipelines are up to date.

DVC status: PASS in 0.9s


CompletedProcess(args=['C:\\Users\\RTX\\.cache\\codex-runtimes\\codex-primary-runtime\\dependencies\\python\\python.exe', '-m', 'dvc', 'status'], returncode=0, stdout='Data and pipelines are up to date.\n')

## 3. CI-Style Quality Gates

In [5]:
run_command(
    "Compile Python files",
    [PYTHON, "-X", "utf8", "-m", "compileall", "src", "scripts", "tests", "app.py"],
    timeout=300,
)
run_command(
    "Ruff lint",
    [PYTHON, "-m", "ruff", "check", "src", "scripts", "tests", "app.py"],
    timeout=300,
)
run_command(
    "Pytest with coverage",
    [
        PYTHON,
        "-m",
        "pytest",
        "tests",
        "--cov=src",
        "--cov-report=term-missing",
        "--cov-fail-under=70",
        "-p",
        "no:cacheprovider",
    ],
    timeout=600,
)


Compile Python files
$ C:\Users\RTX\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\python.exe -X utf8 -m compileall src scripts tests app.py
Listing 'src'...
Listing 'scripts'...
Listing 'tests'...

Compile Python files: PASS in 0.1s

Ruff lint
$ C:\Users\RTX\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\python.exe -m ruff check src scripts tests app.py
All checks passed!

Ruff lint: PASS in 0.1s

Pytest with coverage
$ C:\Users\RTX\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\python.exe -m pytest tests --cov=src --cov-report=term-missing --cov-fail-under=70 -p no:cacheprovider
============================= test session starts =============================
platform win32 -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0
rootdir: C:\Users\RTX\Downloads\Advanced DL
configfile: pyproject.toml
plugins: anyio-4.13.0, hydra-core-1.3.2, cov-7.1.0
collected 18 items

tests\test_bm25_evaluation_integration.py .                              [  5

CompletedProcess(args=['C:\\Users\\RTX\\.cache\\codex-runtimes\\codex-primary-runtime\\dependencies\\python\\python.exe', '-m', 'pytest', 'tests', '--cov=src', '--cov-report=term-missing', '--cov-fail-under=70', '-p', 'no:cacheprovider'], returncode=0, stdout='============================= test session starts =============================\nplatform win32 -- Python 3.12.13, pytest-9.0.3, pluggy-1.6.0\nrootdir: C:\\Users\\RTX\\Downloads\\Advanced DL\nconfigfile: pyproject.toml\nplugins: anyio-4.13.0, hydra-core-1.3.2, cov-7.1.0\ncollected 18 items\n\ntests\\test_bm25_evaluation_integration.py .                              [  5%]\ntests\\test_bm25_retriever.py ........                                    [ 50%]\ntests\\test_evaluation.py ....                                            [ 72%]\ntests\\test_rag_pipeline.py ...                                           [ 88%]\ntests\\test_semantic_retriever.py ..                                      [100%]\n\n=============================== t

## 4. Rerun Retrieval, Evaluation, Semantic Search, and RAG

In [6]:
run_command("Jomana BM25 demo", [PYTHON, "-X", "utf8", "scripts/run_bm25_demo.py"], timeout=300)
run_command("Abdallah BM25 evaluation", [PYTHON, "-X", "utf8", "scripts/run_bm25_evaluation.py"], timeout=300)
run_command("Kadry semantic evaluation", [PYTHON, "-X", "utf8", "scripts/run_semantic_evaluation.py"], timeout=1200)
run_command("Judy RAG demo", [PYTHON, "-X", "utf8", "scripts/run_rag_demo.py"], timeout=1200)


Jomana BM25 demo
$ C:\Users\RTX\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\python.exe -X utf8 scripts/run_bm25_demo.py
Loaded documents: 1500
Average BM25 document length: 84.92 tokens
Exported top-10 results for 5 queries to C:\Users\RTX\Downloads\Advanced DL\results\bm25_sample_top10.csv

Query: Why are micro average precision recall and F1 equal in multiclass classification?
Top hit: [datascience] Micro Average vs Macro average Performance in a Multiclass classification setting (48.493982)
Citation: https://datascience.stackexchange.com/questions/15989/micro-average-vs-macro-average-performance-in-a-multiclass-classification-settin

Query: What is the difference between artificial intelligence and machine learning?
Top hit: [datascience] Difference between machine learning and artificial intelligence (22.033671)
Citation: https://datascience.stackexchange.com/questions/19077/difference-between-machine-learning-and-artificial-intelligence

Query: What are deconv

CompletedProcess(args=['C:\\Users\\RTX\\.cache\\codex-runtimes\\codex-primary-runtime\\dependencies\\python\\python.exe', '-X', 'utf8', 'scripts/run_rag_demo.py'], returncode=0, stdout='Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.\n\nLoading weights:   0%|          | 0/103 [00:00<?, ?it/s]\nLoading weights: 100%|██████████| 103/103 [00:00<00:00, 2582.99it/s]\nJudy RAG integration demo\nDataset: C:\\Users\\RTX\\Downloads\\Advanced DL\\data\\stacklite_questions.csv\nDocuments: 1500\nRetriever mode: hybrid\nGenerator mode: extractive\nQuestions answered: 5\nAnswers CSV: C:\\Users\\RTX\\Downloads\\Advanced DL\\results\\rag_sample_answers.csv\nContexts CSV: C:\\Users\\RTX\\Downloads\\Advanced DL\\results\\rag_retrieved_contexts.csv\nReport: C:\\Users\\RTX\\Downloads\\Advanced DL\\reports\\rag_integration_report.md\n\nQuestion: Why are micro average precision recall and F1 equal in multiclass classif

## 5. UI and Citation-Quality Smoke Checks

In [7]:
ui_check = """
import app

demo = app.build_demo()
answer, contexts, citations, status = app.answer_question(
    "What is the dying ReLU problem in neural networks?",
    app.BM25_RETRIEVER_MODE,
    app.DEFAULT_GENERATOR_MODE,
    10,
    3,
)

assert type(demo).__name__ == "Blocks"
assert contexts
assert citations
print(status)
print(f"contexts: {len(contexts)}")
print(f"citations_present: {bool(citations)}")
print(answer[:250])
"""

run_command("Gradio UI callback smoke test", [PYTHON, "-X", "utf8", "-c", ui_check], timeout=300)

citation_questions_path = PROJECT_ROOT / "evaluation" / "citation_quality_questions.json"
citation_report_path = PROJECT_ROOT / "reports" / "citation_quality_examples.md"
citation_questions = json.loads(citation_questions_path.read_text(encoding="utf-8"))

if not 5 <= len(citation_questions) <= 10:
    raise AssertionError("Citation-quality milestone should include 5 to 10 evaluation questions.")
if not citation_report_path.exists():
    raise FileNotFoundError("Missing reports/citation_quality_examples.md")

print(f"Citation-quality questions: {len(citation_questions)}")
print(f"Citation-quality report: {citation_report_path.relative_to(PROJECT_ROOT)}")


Gradio UI callback smoke test
$ C:\Users\RTX\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\python.exe -X utf8 -c 
import app

demo = app.build_demo()
answer, contexts, citations, status = app.answer_question(
    "What is the dying ReLU problem in neural networks?",
    app.BM25_RETRIEVER_MODE,
    app.DEFAULT_GENERATOR_MODE,
    10,
    3,
)

assert type(demo).__name__ == "Blocks"
assert contexts
assert citations
print(status)
print(f"contexts: {len(contexts)}")
print(f"citations_present: {bool(citations)}")
print(answer[:250])

Used BM25 only with 3 cited contexts.
contexts: 3
citations_present: True
The retrieved StackLite evidence points to the following cited source passages as the best grounding for the answer.

[1] What is the "dying ReLU" problem in neural networks?: What is the "dying ReLU" problem in neural networks? What is the "dying Re

Gradio UI callback smoke test: PASS in 4.2s
Citation-quality questions: 5
Citation-quality report: reports\citation_qua

## 6. Final Summary

In [8]:
print("\nFULL PROJECT RERUN SUMMARY")
print("=" * 88)
for result in RUN_RESULTS:
    print(f"{result['status']:>4} | {result['seconds']:>6}s | {result['step']}")

required_outputs = [
    "results/bm25_sample_top10.csv",
    "results/bm25_evaluation_per_query.csv",
    "results/bm25_evaluation_top10.csv",
    "results/semantic_vs_bm25_comparison.csv",
    "results/semantic_evaluation_top10.csv",
    "results/rag_sample_answers.csv",
    "results/rag_retrieved_contexts.csv",
    "reports/bm25_evaluation_report.md",
    "reports/semantic_search_report.md",
    "reports/rag_integration_report.md",
    "reports/citation_quality_examples.md",
]

missing_outputs = []
print("\nGenerated/verified outputs:")
for relative_path in required_outputs:
    path = PROJECT_ROOT / relative_path
    if path.exists():
        print(f"OK      {relative_path}")
    else:
        missing_outputs.append(relative_path)
        print(f"MISSING {relative_path}")

if missing_outputs:
    raise AssertionError(f"Missing expected outputs: {missing_outputs}")

print("\nAll implemented project milestones reran successfully.")


FULL PROJECT RERUN SUMMARY
PASS |    0.9s | DVC status
PASS |    0.1s | Compile Python files
PASS |    0.1s | Ruff lint
PASS |    2.5s | Pytest with coverage
PASS |    0.5s | Jomana BM25 demo
PASS |    0.5s | Abdallah BM25 evaluation
PASS |   49.5s | Kadry semantic evaluation
PASS |   49.1s | Judy RAG demo
PASS |    4.2s | Gradio UI callback smoke test

Generated/verified outputs:
OK      results/bm25_sample_top10.csv
OK      results/bm25_evaluation_per_query.csv
OK      results/bm25_evaluation_top10.csv
OK      results/semantic_vs_bm25_comparison.csv
OK      results/semantic_evaluation_top10.csv
OK      results/rag_sample_answers.csv
OK      results/rag_retrieved_contexts.csv
OK      reports/bm25_evaluation_report.md
OK      reports/semantic_search_report.md
OK      reports/rag_integration_report.md
OK      reports/citation_quality_examples.md

All implemented project milestones reran successfully.
